In [8]:
#1.라이브러리 임포트 및 경고 제어 
import warnings
# 모델 로딩 시 발생하는 자잘한 경고 메시지(버전 차이 등)를 숨겨서 출력창을 깔끔하게 유지합니다.
warnings.filterwarnings('ignore') 
# 허깅페이스에서 제공하는 파이프라인을 가져옵니다.
from transformers import pipeline

print("=" * 60)
print(" 6개 모델(A그룹 3개, B그룹 3개) 동시 다운로드 및 로딩 중...")
print("로딩 실패 모델은 자동으로 넘깁니다.")
print("=" * 60)

# 2. 테스트할 모델 리스트
# A그룹 : 주로 뉴스 기사 / 정제된 문장 분석에 특화된 모델
dict_A = {
    "KR-FinBERT (금융 뉴스 모델)": "snunlp/KR-FinBert-SC",
    "KLUE-RoBERTa (일반 뉴스 특화 모델)": "mrm8488/klue-roberta-small-finetuned-nsmc",
    "KoBERT (SKT 모델)": "sangrimlee/bert-base-multilingual-cased-nsmc"
}

# B그룹 : 주로 '댓글', '비속어' 분석에 특화된 모델
dict_B = {
    "KoELECTRA-v2 (댓글 특화 모델)": "monologg/koelectra-small-v2-nsmc",
    "KoELECTRA-v3 (범용적 모델)": "jaehyeong/koelectra-base-v3-generalized-sentiment-analysis",
    "Korean-Sentiment (댓글 특화 모델)": "matthewburke/korean_sentiment"
}

#로딩에 성공한 모델들만 따로 담아둘 빈 딕셔너리
active_A = {}
active_B = {}

#3. 모델 로딩
# A그룹 
for name, path in dict_A.items():
    try:
        # 텍스트 분류 모델을 불러오며 
        # device=-1 은 Cpu를 사용 GPU면 0으로 변경
        active_A[name] = pipeline("text-classification", model=path, device=-1)
        print(f" 로딩 성공: {name}")
    except:
        print(f" 로딩 실패 (건너뜀): {name}")

print("-" * 30)

# B그룹 
for name, path in dict_B.items():
    try:
        active_B[name] = pipeline("text-classification", model=path, device=-1)
        print(f" 로딩 성공: {name}")
    except:
        print(f" 로딩 실패 (건너뜀): {name}")

print("=" * 60)

# 4. 테스트할 텍스트 18가지 (기사 6개 + 댓글 12개)
test_texts = [
    # 바리에이션 1: 부정 기사 + 긍정/긍정 댓글 
    "[뉴스기사] 지난해 4분기 반도체 부문 2조원대 영업손실… 최악의 보릿고개 통과 중",
    "[댓글반응] 원래 실적 최악일 때가 주가는 바닥인 거 모름? 이제 악재 다 반영됐고 올라갈 일만 남았네 ㅋㅋㅋ",
    "[댓글반응] 어차피 사이클 산업이라 버티면 올라옴. 쌀 때 부지런히 모아갑니다.",

    # 바리에이션 2: 부정 기사 + 긍정/부정 댓글
    "[뉴스기사] 엔비디아 블랙웰 출시 지연 우려에… SK하이닉스 등 국내 HBM 밸류체인 일제히 급락세",
    "[댓글반응] 어차피 HBM 수요는 견조함. 단기 노이즈일 뿐이니 지금 줍줍해서 수량 늘려가면 승리자 됨.",
    "[댓글반응] 맨날 미장 핑계 대면서 내리꽂네. 반등 주면 다 던지고 나갑니다.",
    
    # 바리에이션 3: 부정 기사 + 부정/부정 댓글
    "[뉴스기사] 모건스탠리 '반도체 겨울이 다가온다'… D램 가격 하락 전망에 반도체주 투심 꽁꽁",
    "[댓글반응] 겨울 정도가 아니라 빙하기구만. 외인 기관 다 던지는데 개미들만 총알받이 하네 ㅠㅠ",
    "[댓글반응] 고점에 물려서 이제 물 탈 돈도 없다...",

  # 바리에이션 4: 긍정 기사 + 긍정/긍정 댓글
    "[뉴스기사] 한국 반도체 수출액 100억 달러 재돌파… AI 메모리 수요 폭발에 역대급 호황기 진입",
    "[댓글반응] 영업이익률 개선되는 거 보면 펀더멘털 진짜 탄탄해졌네요. 장기 투자하기 딱 좋습니다.",
    "[댓글반응] 숏충이들 다 한강 갔냐? ㅋㅋㅋ 외국인 형님들 쌍끌이 매수 들어오는 거 보소 가즈아!!!",

    # 바리에이션 5: 긍정 기사 + 부정/긍정 댓글
    "[뉴스기사] 삼성전자, 5세대 HBM(HBM3E) 8단 엔비디아 퀄테스트 통과… 4분기부터 본격 공급 돌입",
    "[댓글반응] 며칠 전에 찌라시 돌아서 이미 선반영 다 끝났음. 고점에서 설거지 당하지 마라.",
    "[댓글반응] 드디어 불확실성 해소됐네!! 10만전자 다시 가보자 꽉 잡아라!!",
    
    # 바리에이션 6: 긍정 기사 + 부정/부정 댓글 
    "[뉴스기사] 파운드리 적자폭 크게 축소… 내년 하반기 2나노 양산으로 TSMC 맹추격 예고",
    "[댓글반응] 맨날 추격만 하고 언제 잡는데 ㅋㅋㅋ 말만 번지르르하네.",
    "[댓글반응] 적자폭 줄인 게 자랑인가... 결국 아직도 마이너스라는 건데 이게 진짜 호재 맞음?"
    
]

# 바리에이션 설명용 딕셔너리
var_desc = {
    1: "부정 기사 + 긍정/긍정 댓글",
    2: "부정 기사 + 긍정/부정 댓글",
    3: "부정 기사 + 부정/부정 댓글",
    4: "긍정 기사 + 긍정/긍정 댓글",
    5: "긍정 기사 + 부정/긍정 댓글",
    6: "긍정 기사 + 부정/부정 댓글"
}

j = 1

# 5. 결과값 통일화 함수 정의
# 허깅페이스 모델들 마다 긍정 부정 출력할 때 1/0 or Positive Negative라고 나오는데
# 그걸 통일화해주는 작업입니다.
def format_label(label, score):
    lbl = str(label).lower()
    if '1' in lbl or 'pos' in lbl:
        return f" 긍정 ({score*100:5.1f}%)"
    elif '0' in lbl or 'neg' in lbl:
        return f" 부정 ({score*100:5.1f}%)"
    else:
        return f" 중립 ({score*100:5.1f}%)"

# 6. 모델 비교 및 실제 테스트 결과 출력
for i, text in enumerate(test_texts):

    if i % 3 == 0: 
        print("\n"*2)
        print(f"바리에이션 {j} {var_desc[j]}\n")
       
        j = j+1


    print(f"\n 테스트 문장 {i+1}: {text}")
    print("-" * 60)
    
    #A그룹 긴텍스트 메인
    print("[A그룹 / 뉴스 특화 모델들]")
    #그룹 안전장치 모델 로딩시 로딩이 전부 안되면 파악하려고 넣어뒀습니다.
    if not active_A:
        print("  - 로딩된 A그룹 모델이 없습니다.")
        # 아까 만든 Format_label 함수에 결과값을 넣어 출력
    for name, analyzer in active_A.items():
        res = analyzer(text)[0]
        print(f"  ▪️ {name:<25} : {format_label(res['label'], res['score'])}")
        
    #B그룹 짧은 텍스트 메인
    print("\n[B그룹 / 댓글 특화 모델들]")
    if not active_B:
        print("  - 로딩된 B그룹 모델이 없습니다.")
    for name, analyzer in active_B.items():
        res = analyzer(text)[0]
        print(f"  ▪️ {name:<25} : {format_label(res['label'], res['score'])}")
    print("=" * 60)

 6개 모델(A그룹 3개, B그룹 3개) 동시 다운로드 및 로딩 중...
로딩 실패 모델은 자동으로 넘깁니다.


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 43589.01it/s]


 로딩 성공: KR-FinBERT (금융 뉴스 모델)
 로딩 실패 (건너뜀): KLUE-RoBERTa (일반 뉴스 특화 모델)


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 43720.12it/s]


 로딩 성공: KoBERT (SKT 모델)
------------------------------
 로딩 실패 (건너뜀): KoELECTRA-v2 (댓글 특화 모델)


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 6225.71it/s]


 로딩 성공: KoELECTRA-v3 (범용적 모델)


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 49284.18it/s]


 로딩 성공: Korean-Sentiment (댓글 특화 모델)



바리에이션 1 부정 기사 + 긍정/긍정 댓글


 테스트 문장 1: [뉴스기사] 지난해 4분기 반도체 부문 2조원대 영업손실… 최악의 보릿고개 통과 중
------------------------------------------------------------
[A그룹 / 뉴스 특화 모델들]
  ▪️ KR-FinBERT (금융 뉴스 모델)     :  부정 (100.0%)
  ▪️ KoBERT (SKT 모델)           :  부정 ( 99.6%)

[B그룹 / 댓글 특화 모델들]
  ▪️ KoELECTRA-v3 (범용적 모델)     :  부정 ( 98.0%)
  ▪️ Korean-Sentiment (댓글 특화 모델) :  부정 ( 89.6%)

 테스트 문장 2: [댓글반응] 원래 실적 최악일 때가 주가는 바닥인 거 모름? 이제 악재 다 반영됐고 올라갈 일만 남았네 ㅋㅋㅋ
------------------------------------------------------------
[A그룹 / 뉴스 특화 모델들]
  ▪️ KR-FinBERT (금융 뉴스 모델)     :  부정 ( 98.3%)
  ▪️ KoBERT (SKT 모델)           :  부정 ( 98.4%)

[B그룹 / 댓글 특화 모델들]
  ▪️ KoELECTRA-v3 (범용적 모델)     :  긍정 ( 69.5%)
  ▪️ Korean-Sentiment (댓글 특화 모델) :  긍정 ( 88.8%)

 테스트 문장 3: [댓글반응] 어차피 사이클 산업이라 버티면 올라옴. 쌀 때 부지런히 모아갑니다.
------------------------------------------------------------
[A그룹 / 뉴스 특화 모델들]
  ▪️ KR-FinBERT (금융 뉴스 모델)     :  중립 (100.0%)
  ▪️ KoBERT (SKT 모델)           :  부정 ( 81.1%)

[B그룹 